# 📘 Importing

In [1]:
import os
import ast
import operator
import pynvml

import pandas as pd

from dotenv import load_dotenv
from operator import itemgetter
from codecarbon import track_emissions
from IPython.display import Image, display

from time import sleep

C:\Users\Administrador\AppData\Local\Temp\ipykernel_61544\1072440278.py:4: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml


In [2]:
from pydantic import BaseModel, Field
from typing_extensions import TypedDict, List, Literal, Annotated

In [3]:
from langgraph.graph import StateGraph, START, END

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings

from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langgraph.prebuilt import create_react_agent, ToolNode
from langgraph_supervisor import create_supervisor

from langchain.tools.retriever import create_retriever_tool

from langchain_core.tools import tool, Tool

from langchain_core.output_parsers import StrOutputParser

from langchain.vectorstores import FAISS
from langchain.schema import Document

In [4]:
from deepeval import evaluate
from deepeval.dataset import EvaluationDataset
from deepeval.metrics import (
    AnswerRelevancyMetric, 
    FaithfulnessMetric,
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric
)
from deepeval.test_case import LLMTestCase
from deepeval.models import OllamaModel

# ⚙️ Settings

In [5]:
llm_retriever = ChatOllama(model="gpt-oss:latest")
llm_supervisor = ChatOllama(model="gpt-oss:latest")

llm_eval = OllamaModel(model="gemma3:27b")

In [6]:
embeddings_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

In [7]:
answer_relevancy = AnswerRelevancyMetric(threshold=0.5, model=llm_eval)
faithfulness = FaithfulnessMetric(threshold=0.5, model=llm_eval)
contextual_relevancy = ContextualRelevancyMetric(threshold=0.5, model=llm_eval)
contextual_precision = ContextualPrecisionMetric(threshold=0.5, model=llm_eval)
contextual_recall = ContextualRecallMetric(threshold=0.5, model=llm_eval)

In [8]:
def format_docs(docs: List[Document]) -> str:
    return "\n\n".join(doc.page_content for doc in docs)

In [9]:
class RetrievalMonitor:
    def __init__(self):
        self.retrieved_docs = []

    def _create_monitored_retriever_tool(self, retriever, name, description):
        def monitored_retrieval(query: str) -> str:
            docs = retriever.invoke(query)

            self.retrieved_docs.extend(docs)
            
            return format_docs(docs)
        
        return Tool(
            name=name, 
            description=description, 
            func=monitored_retrieval
        )

    def get_tool(self, retriever, name, description):
        return self._create_monitored_retriever_tool(
            retriever=retriever,
            name=name,
            description=description
        )

In [10]:
monitor_dev = RetrievalMonitor()
monitor_eng = RetrievalMonitor()

In [11]:
vectorstore_dev = FAISS.load_local(
    "../../data/vectorstore/faiss_embeddings_developers", 
    embeddings_model, 
    allow_dangerous_deserialization=True
)

retriever_dev = vectorstore_dev.as_retriever(search_kwargs={"k": 20})

retriever_dev_tool = monitor_dev.get_tool(
    retriever=retriever_dev,
    name="developer-base-retriever-tool",
    description="""
    A tool to retrieve relevant information from a vectorstore 
    of developer-related documents for industrial network 
    technical requests of PROFIBUS, to answer:
    - Protocol specs: data-link, transport or application layers
    - Message formats, telegram structure, data types
    - Device profiles, GSD/EDS details, FSCP (Functional Safety)
    - Parameterization, timing calculations, test procedures
    """
)

In [12]:
vectorstore_eng = FAISS.load_local(
    "../../data/vectorstore/faiss_embeddings_engineers", 
    embeddings_model, 
    allow_dangerous_deserialization=True
)

retriever_eng = vectorstore_eng.as_retriever(search_kwargs={"k": 20})

retriever_eng_tool = monitor_eng.get_tool(
    retriever=retriever_eng,
    name="engineer-base-retriever-tool",
    description="""
    A tool to retrieve relevant information from a vectorstore 
    of engineer-related documents for industrial network 
    technical requests of PROFIBUS, to answer:
    - Physical installation: cabling, shielding, grounding
    - EMI/EMC mitigation, cable routing, connector assemblies
    - Commissioning & maintenance: troubleshooting, oscilloscope usage
    - Hazardous areas, equipotential bonding, field documentation
    """
)

In [13]:
developer_retriever_agent = create_react_agent(
    model=llm_retriever,
    tools=[
        retriever_dev_tool,
    ],
    name="developer_retriever_agent",
    prompt="""
    You are an expert technical analyst specializing in industrial communication standards like PROFIBUS. 
    Your primary function is to provide comprehensive and detailed answers based on technical documents.

    # CORE TASK
    A user will ask a question. Your tools will retrieve multiple, relevant text chunks from technical specifications. These chunks may be repetitive, 
    contain formatting artifacts (like markdown tables), or present different values for the same parameter under different conditions.

    Your job is to meticulously analyze and synthesize ALL the provided information into a single, coherent, and well-structured answer.

    # INSTRUCTIONS
    1.  **Synthesize, Don't Just Report:** Do not just find one relevant fact and report it. Your value is in combining details from all retrieved chunks.
    2.  **Explain Nuances:** If you find different values for the same technical parameter (e.g., "bit cell jitter"), you MUST explain the context for each. 
    For example: "For transmitted signals, the specification is X, while for received signals, the tolerance is Y. Furthermore, for optical interfaces, the value is Z."
    3.  **Structure for Clarity:** Format your final response using clear paragraphs, bullet points, or lists to make the technical information easy to understand.
    4.  **Be Grounded:** Base your entire answer strictly on the information provided by the tools. Do not invent or assume details.
    5.  **Ignore Artifacts:** Disregard any messy formatting (like `<br>`, `|`, `<span>`) in the context and focus solely on the technical content.
    """
)

In [14]:
engineer_retriever_agent = create_react_agent(
    model=llm_retriever,
    tools=[
        retriever_eng_tool,
    ],
    name="engineer_retriever_agent",
    prompt="""
    You are an expert field engineer and PROFIBUS network specialist. 
    Your primary function is to provide comprehensive and practical guidance on installation, commissioning, and troubleshooting based on technical documents.

    # CORE TASK
    A user will ask a question. Your tools will retrieve multiple, relevant text chunks from technical guides and best-practice documents. These chunks may be repetitive, 
    contain formatting artifacts, or present different recommendations for the same problem.

    Your job is to meticulously analyze and synthesize ALL the provided information into a single, coherent, and actionable answer.

    # INSTRUCTIONS
    1.  **Synthesize, Don't Just Report:** Do not just find one relevant fact and report it. Your value is in combining details from all retrieved chunks to form a complete picture.
    2.  **Explain Nuances and Context:** If you find different recommendations for the same topic (e.g., grounding methods), you MUST explain the context for each. 
    For example: "For standard installations, direct shield grounding is used, but in environments with potential ground loops, capacitive grounding is the required solution."
    3.  **Structure for Clarity:** Format your final response using clear paragraphs, bullet points, or step-by-step lists to make the technical guidance easy to follow.
    4.  **Be Grounded:** Base your entire answer strictly on the information provided by the tools. Do not invent or assume details.
    5.  **Ignore Artifacts:** Disregard any messy formatting (like `<br>`, `|`, `<span>`) in the context and focus solely on the practical engineering content.
    """
)

In [15]:
hierarchical_supervisor = create_supervisor(
    model=llm_supervisor,
    agents=[
        developer_retriever_agent, 
        engineer_retriever_agent
    ],
    prompt=""""
    # ROLE
    You are an expert supervisor agent responsible for routing technical PROFIBUS network queries.

    # PRIMARY OBJECTIVE
    Your task is to analyze a user's query and decide which of the two specialist agents is best equipped to answer it. 
    You must route the query to the correct agent based on their defined specializations.

    # AGENT SPECIALIZATIONS

    **1. developer_retriever_agent:**
    These documents address the IEC 61158 and IEC 61784 standards, focusing specifically 
    on their applications for industrial communication networks, such as fieldbuses. The 
    texts detail the structure and behavior of protocols across various layers, including 
    the physical layer (PhL), the data link layer (DLL), and the application layer (AL). 
    Information is provided on data transmission, master-slave device management, and 
    security and performance configurations, with descriptions of service primitives, 
    class attributes, and finite state machines that govern the communication flow and 
    interactions between network components. There is a strong emphasis on interoperability 
    and conformity through communication profiles and cross-references to other IEC standards.

    **2. engineer_retriever_agent:**
    The sources provide a comprehensive overview of PROFIBUS systems, 
    with a focus on electromagnetic compatibility (EMC), functional grounding, and 
    shielding. They detail various interference coupling mechanisms, such as conductive, 
    capacitive, and inductive coupling, and the common sources of disturbances in 
    industrial environments. The documents also present recommendations for the design 
    and commissioning of networks, including the use of TN-S grounding systems and coarse-mesh 
    equipotential bonding networks (CBNs). Furthermore, the sources address measurement and 
    troubleshooting methods, such as shield current and oscilloscope measurements, and emphasize 
    the importance of documentation and quality certification to ensure system reliability.

    # INSTRUCTIONS
    1.  Carefully analyze the user's question.
    2.  Compare the core concepts of the question against the agent specializations.
    3.  Do not add or assume any information that is not in the user's query.
    4.  Assign work to one agent at a time, do not call agents in parallel.
    5.  Do not do any work yourself.

    # IMPORTANT
    First, rewrite the user query in english and to be more specific to PROFIBUS technology.
    If the response from the agent dont looks like an answer to the user's question,
    you must route the query to the other agent. The final response must be the answer by
    the agent you decide to call. Your final response will be only the response without put
    the question or what agent gave you the response
    """
)

workflow = hierarchical_supervisor.compile()

In [16]:
display(Image(workflow.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 502.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

# 📊 Evaluation

In [17]:
import time
from codecarbon import EmissionsTracker

In [18]:
qna_dev = pd.read_csv("../../data/QnA_dev.csv")
qna_eng = pd.read_csv("../../data/QnA_eng.csv")

In [ ]:
qna_dev["response"] = ""
qna_dev["duration"] = 0.0
qna_dev["emission_kg"] = 0.0

qna_dev["retrieved_context"] = [[] for _ in range(len(qna_dev))]

tracker = EmissionsTracker(project_name="02_QnA_dev")

tracker.start()

for i, row in qna_dev.iterrows():
    monitor_dev.retrieved_docs.clear()

    tracker.start_task(f"Question id: {row['id']}")
    start_time = time.perf_counter()

    response = workflow.invoke({
        "messages": [
            {
                "role": "user",
                "content": row["question"]
            }
        ]
    }, {"recursion_limit": 50})

    end_time = time.perf_counter()
    emissions_data = tracker.stop_task()

    retrieved_context = [doc.page_content for doc in monitor_dev.retrieved_docs]

    duration = end_time - start_time

    qna_dev.at[i, "response"] = response["messages"][-1].content
    qna_dev.at[i, "duration"] = duration
    qna_dev.at[i, "emission_kg"] = emissions_data.emissions
    qna_dev.at[i, "retrieved_context"] = retrieved_context

[codecarbon WARNING @ 08:52:10] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 08:52:10] [setup] RAM Tracking...
[codecarbon INFO @ 08:52:10] [setup] CPU Tracking...
[codecarbon WARNING @ 08:52:11] We saw that you have a Intel(R) Core(TM) i9-14900K but we don't know it. Please contact us.
[codecarbon WARNING @ 08:52:11] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 08:52:11] CPU Model on constant consumption mode: Intel(R) Core(TM) i9-14900K
[codecarbon WARNING @ 08:52:11] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 08:52:11] [setup] GPU Tracking...
[codecarbon INFO @ 08:52:11] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 08:52:11] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global

[codecarbon INFO @ 09:49:08] Energy consumed for RAM : 0.051528 kWh. RAM Power : 54.0 W
[codecarbon INFO @ 09:49:08] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 09:49:08] Energy consumed for All CPU : 0.040562 kWh
[codecarbon INFO @ 09:49:08] Energy consumed for all GPUs : 0.262916 kWh. Total GPU Power : 256.048691440615 W
[codecarbon INFO @ 09:49:08] 0.355006 kWh of electricity used since the beginning.
[codecarbon INFO @ 09:49:19] Energy consumed for RAM : 0.051529 kWh. RAM Power : 54.0 W
[codecarbon INFO @ 09:49:19] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 09:49:19] Energy consumed for All CPU : 0.040563 kWh
[codecarbon INFO @ 09:49:19] Energy consumed for all GPUs : 0.262997 kWh. Total GPU Power : 74.7683795612758 W
[codecarbon INFO @ 09:49:19] 0.355089 kWh of electricity used since the beginning.
[codecarbon INFO @ 09:49:23] Energy consumed for RAM : 0.051753 kWh. RAM Power : 54.

In [ ]:
qna_dev.to_csv("../../data/02_QnA_dev.csv")

[codecarbon INFO @ 09:49:34] Energy consumed for RAM : 0.051754 kWh. RAM Power : 54.0 W
[codecarbon INFO @ 09:49:34] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 09:49:34] Energy consumed for All CPU : 0.040740 kWh
[codecarbon INFO @ 09:49:34] Energy consumed for all GPUs : 0.263082 kWh. Total GPU Power : 20.615336742034295 W
[codecarbon INFO @ 09:49:34] 0.355577 kWh of electricity used since the beginning.
[codecarbon INFO @ 09:49:38] Energy consumed for RAM : 0.051978 kWh. RAM Power : 54.0 W
[codecarbon INFO @ 09:49:38] Delta energy consumed for CPU with constant : 0.000177 kWh, power : 42.5 W
[codecarbon INFO @ 09:49:38] Energy consumed for All CPU : 0.040917 kWh
[codecarbon INFO @ 09:49:38] Energy consumed for all GPUs : 0.263122 kWh. Total GPU Power : 19.541334568311424 W
[codecarbon INFO @ 09:49:38] 0.356018 kWh of electricity used since the beginning.


In [19]:
qna_eng["response"] = ""
qna_eng["duration"] = 0.0
qna_eng["emission_kg"] = 0.0

qna_eng["retrieved_context"] = [[] for _ in range(len(qna_eng))]

tracker = EmissionsTracker(project_name="02_QnA_eng")

tracker.start()

for i, row in qna_eng.iterrows():
    monitor_eng.retrieved_docs.clear()

    tracker.start_task(f"Question id: {row['id']}")
    start_time = time.perf_counter()

    response = workflow.invoke({
        "messages": [
            {
                "role": "user",
                "content": row["question"]
            }
        ]
    })

    end_time = time.perf_counter()
    emissions_data = tracker.stop_task()

    retrieved_context = [doc.page_content for doc in monitor_eng.retrieved_docs]

    duration = end_time - start_time

    qna_eng.at[i, "response"] = response["messages"][-1].content
    qna_eng.at[i, "duration"] = duration
    qna_eng.at[i, "emission_kg"] = emissions_data.emissions
    qna_eng.at[i, "retrieved_context"] = retrieved_context

[codecarbon WARNING @ 10:50:13] Multiple instances of codecarbon are allowed to run at the same time.
[codecarbon INFO @ 10:50:13] [setup] RAM Tracking...
[codecarbon INFO @ 10:50:13] [setup] CPU Tracking...
[codecarbon WARNING @ 10:50:15] We saw that you have a Intel(R) Core(TM) i9-14900K but we don't know it. Please contact us.
[codecarbon WARNING @ 10:50:15] No CPU tracking mode found. Falling back on estimation based on TDP for CPU. 
 Windows OS detected: Please install Intel Power Gadget to measure CPU

[codecarbon INFO @ 10:50:15] CPU Model on constant consumption mode: Intel(R) Core(TM) i9-14900K
[codecarbon WARNING @ 10:50:15] No CPU tracking mode found. Falling back on CPU constant mode.
[codecarbon INFO @ 10:50:15] [setup] GPU Tracking...
[codecarbon INFO @ 10:50:15] Tracking Nvidia GPU via pynvml
[codecarbon INFO @ 10:50:15] The below tracking methods have been set up:
                RAM Tracking Method: RAM power estimation model
                CPU Tracking Method: global

In [20]:
qna_eng.to_csv("../../data/02_QnA_eng.csv")